# Source Model and Target Adaptation

This notebook trains a neural feature encoder on UNSW traffic and uses a small set of verified CIC records to train an adapted intrusion detection classifier.

It compares random selection, standard reinforcement learning (RL), and diversity-aware RL using the same **1,000-label budget** across three seeds.

**Before running:** Run `01_data_preparation.ipynb` once to create the sampled CSV files and feature schema. This notebook reads those saved files directly, so it does not depend on variables from another Colab session. It does **not** read the separate 20,000-record holdout.

In [ ]:
from pathlib import Path
from collections import Counter
import copy
import json

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.distributions import Categorical
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, average_precision_score, accuracy_score

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Outside Colab: change BASE below to your local dataset folder.')

BASE = Path('/content/drive/MyDrive/datanew/a4_analysis')
RUN_DIR = BASE / 'reproduction_runs' / 'source_adaptation'
RUN_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CSV = BASE / 'UNSW_100k.csv'
TARGET_CSV = BASE / 'CIC_100k.csv'
SCHEMA_FILE = BASE / 'csc786_feature_schema.json'
SOURCE_CHECKPOINT = BASE / 'cell_44C_multiclass_source.pt'

for path in (SOURCE_CSV, TARGET_CSV, SCHEMA_FILE):
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path.name}. Run the data preparation notebook first.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(2)
print('Device:', device)
print('Results will be saved to:', RUN_DIR)

## 1. Load the features

We use the same 43 features defined in the data-preparation notebook. The features describe flow activity without using IP addresses or attack labels as model inputs.

In [ ]:
with SCHEMA_FILE.open('r', encoding='utf-8') as f:
    schema = json.load(f)

raw_features = schema['raw_features']
base_features = schema['retained_raw_features']
engineered_features = schema['engineered_features']
all_features = schema['final_features_in_order']
assert len(raw_features) == 41 and len(base_features) == 37 and len(all_features) == 43
assert all_features == base_features + engineered_features


def engineer_features(frame):
    result = frame.copy()
    in_bytes = result['IN_BYTES'].clip(lower=0)
    out_bytes = result['OUT_BYTES'].clip(lower=0)
    in_pkts = result['IN_PKTS'].clip(lower=0)
    out_pkts = result['OUT_PKTS'].clip(lower=0)
    total_bytes = in_bytes + out_bytes
    total_pkts = in_pkts + out_pkts

    result['FE_LOG_TOTAL_BYTES'] = np.log1p(total_bytes)
    result['FE_LOG_TOTAL_PKTS'] = np.log1p(total_pkts)
    result['FE_BYTE_ASYMMETRY'] = (in_bytes - out_bytes) / (total_bytes + 1)
    result['FE_PKT_ASYMMETRY'] = (in_pkts - out_pkts) / (total_pkts + 1)
    result['FE_LOG_AVG_PKT_SIZE'] = np.log1p(total_bytes / (total_pkts + 1))
    result['FE_LOG_DURATION'] = np.log1p(
        result['FLOW_DURATION_MILLISECONDS'].clip(lower=0)
    )
    return result

source = engineer_features(pd.read_csv(SOURCE_CSV))
target = engineer_features(pd.read_csv(TARGET_CSV))

for name, frame in [('UNSW', source), ('CIC', target)]:
    missing = set(all_features + ['Label', 'Attack']) - set(frame.columns)
    if missing:
        raise ValueError(f'{name}: missing columns: {sorted(missing)}')
    if not np.isfinite(frame[all_features].to_numpy(dtype=np.float64)).all():
        raise ValueError(f'{name}: non-finite feature values')
    if not frame['Label'].isin([0, 1]).all():
        raise ValueError(f'{name}: unexpected binary labels')
    print(f'{name}: {len(frame):,} records, {len(all_features)} features')

## 2. Split the data and fit preprocessing

We group identical feature records before splitting to reduce duplicate leakage. The source data is split into training, validation, and test sets. CIC data is split into a label-acquisition pool and a development evaluation set.

The numerical scaler is fitted **only on UNSW training records**. An `asinh` transform is used for one extreme-valued feature, followed by the same transformation of standardized values across both datasets.

In [ ]:
source_groups = pd.util.hash_pandas_object(
    source[raw_features], index=False
).to_numpy()
train_idx, remaining_idx = next(GroupShuffleSplit(
    n_splits=1, test_size=0.30, random_state=13
).split(source, groups=source_groups))
val_local, test_local = next(GroupShuffleSplit(
    n_splits=1, test_size=0.50, random_state=13
).split(source.iloc[remaining_idx], groups=source_groups[remaining_idx]))
source_train = source.iloc[train_idx].copy()
source_val = source.iloc[remaining_idx[val_local]].copy()
source_test = source.iloc[remaining_idx[test_local]].copy()

target_groups = pd.util.hash_pandas_object(
    target[raw_features], index=False
).to_numpy()
adapt_idx, eval_idx = next(GroupShuffleSplit(
    n_splits=1, test_size=0.50, random_state=786
).split(target, groups=target_groups))
target_adapt = target.iloc[adapt_idx].copy()
target_eval = target.iloc[eval_idx].copy()

special_feature = all_features.index('SRC_TO_DST_SECOND_BYTES')

def safe_feature_transform(frame):
    x = frame[all_features].to_numpy(dtype=np.float64, copy=True)
    x[:, special_feature] = np.arcsinh(x[:, special_feature])
    return x

source_scaler = StandardScaler().fit(safe_feature_transform(source_train))

def transform_for_model(frame):
    x = source_scaler.transform(safe_feature_transform(frame))
    x = np.arcsinh(x).astype(np.float32)
    if not np.isfinite(x).all():
        raise ValueError('Non-finite model features')
    return x

X_train = transform_for_model(source_train)
X_val = transform_for_model(source_val)
X_adapt = transform_for_model(target_adapt)
X_eval = transform_for_model(target_eval)

# Remove duplicates within the evaluation set and any records also in adaptation.
pool_hash = pd.util.hash_pandas_object(
    target_adapt[all_features], index=False
).to_numpy()
eval_hash = pd.util.hash_pandas_object(
    target_eval[all_features], index=False
).to_numpy()
unique_eval = ~pd.Series(eval_hash).duplicated(keep='first').to_numpy()
eval_mask = unique_eval & ~np.isin(eval_hash, np.unique(pool_hash))
X_eval = X_eval[eval_mask]
target_eval_labels = target_eval.iloc[np.flatnonzero(eval_mask)].copy()

assert X_train.shape[1] == X_adapt.shape[1] == 43
assert len(X_eval) == len(target_eval_labels)
print('UNSW train / val / test:', len(source_train), len(source_val), len(source_test))
print('CIC adaptation / development:', len(target_adapt), len(target_eval_labels))
print('CIC holdout used: No')
joblib.dump(source_scaler, RUN_DIR / 'source_standard_scaler.joblib')

## 3. Train or load the source encoder

The neural network learns a 32-dimensional representation from the 43 input features. It is trained only on UNSW attack categories. Its final checkpoint is chosen using UNSW validation data.

If the completed source checkpoint is present in Drive, we load it. Otherwise, this notebook trains the network and saves a new checkpoint in the reproduction folder.

In [ ]:
CLASS_NAMES = [
    'Benign', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance',
    'DoS', 'Backdoor', 'Analysis', 'Shellcode', 'Worms'
]
class_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}


def source_categories(frame):
    binary = frame['Label'].to_numpy(dtype=int)
    attack = frame['Attack'].astype('string').str.strip().to_numpy(dtype=object)
    names = np.where(binary == 0, 'Benign', attack)
    if pd.isna(names).any() or not set(names).issubset(set(CLASS_NAMES)):
        raise ValueError('Unexpected or missing UNSW attack category')
    return np.array([class_to_id[name] for name in names], dtype=np.int64)


y_train = source_categories(source_train)
y_val = source_categories(source_val)

class RareAwareNet(nn.Module):
    def __init__(self, input_dim=43, n_classes=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 32), nn.GELU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64), nn.GELU(), nn.Linear(64, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(32), nn.Linear(32, n_classes)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z), z


model = RareAwareNet().to(device)
print('UNSW training labels:')
print(pd.Series(y_train).map(dict(enumerate(CLASS_NAMES))).value_counts().to_string())

In [ ]:
LOAD_EXISTING_TEACHER = True


def train_source_encoder(model):
    torch.manual_seed(44)
    np.random.seed(44)
    model = RareAwareNet().to(device)
    x_train = torch.as_tensor(X_train, dtype=torch.float32)
    y_tensor = torch.as_tensor(y_train, dtype=torch.long)

    # Two denoising pretraining epochs.
    pretrain_loader = DataLoader(x_train, batch_size=512, shuffle=True)
    pretrain_optimizer = torch.optim.Adam(
        list(model.encoder.parameters()) + list(model.decoder.parameters()), lr=0.001
    )
    for epoch in range(2):
        model.train()
        losses = []
        for xb in pretrain_loader:
            xb = xb.to(device)
            masked = xb * (torch.rand_like(xb) > 0.15).float()
            reconstruction = model.decoder(model.encoder(masked))
            loss = F.smooth_l1_loss(reconstruction, xb)
            pretrain_optimizer.zero_grad()
            loss.backward()
            pretrain_optimizer.step()
            losses.append(loss.item())
        print(f'Denoising {epoch + 1}/2: {np.mean(losses):.4f}')

    # Larger weights help avoid ignoring rare source attack categories.
    class_counts = np.bincount(y_train, minlength=len(CLASS_NAMES))
    weights = np.clip((class_counts.max() / np.maximum(class_counts, 1)) ** 0.25, 1, 8)
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

    def rare_focal_loss(logits, labels):
        ce = F.cross_entropy(logits, labels, reduction='none')
        p = F.softmax(logits, dim=1)
        pt = p.gather(1, labels.unsqueeze(1)).squeeze(1).clamp(1e-7, 1)
        sw = class_weights[labels]
        return (sw * (1 - pt) ** 1.5 * ce).sum() / sw.sum()

    train_loader = DataLoader(TensorDataset(x_train, y_tensor), batch_size=256, shuffle=True)
    optimizer = torch.optim.AdamW(
        list(model.encoder.parameters()) + list(model.classifier.parameters()),
        lr=0.0007, weight_decay=0.0001
    )
    val_x = torch.as_tensor(X_val, dtype=torch.float32, device=device)
    best_score, best_epoch, best_weights = -1, 0, None
    patience, history = 0, []

    for epoch in range(1, 21):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, _ = model(xb)
            loss = rare_focal_loss(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            optimizer.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            pred = model(val_x)[0].argmax(dim=1).cpu().numpy()
        score = f1_score(
            y_val, pred, labels=np.arange(len(CLASS_NAMES)),
            average='macro', zero_division=0
        )
        history.append({'Epoch': epoch, 'Loss': np.mean(losses), 'Validation_Macro_F1': score})
        if score > best_score + 1e-5:
            best_score, best_epoch = score, epoch
            best_weights = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
        print(f'Epoch {epoch:02d}: source macro-F1 {score:.4f}')
        if patience >= 5:
            break

    model.load_state_dict(best_weights)
    model.eval()
    pd.DataFrame(history).to_csv(RUN_DIR / 'source_training_history.csv', index=False)
    torch.save({
        'state_dict': model.state_dict(), 'class_names': CLASS_NAMES,
        'input_features': all_features, 'best_epoch': best_epoch
    }, RUN_DIR / 'source_encoder_checkpoint.pt')
    print('Source checkpoint saved (epoch', best_epoch, ')')
    return model


if LOAD_EXISTING_TEACHER and SOURCE_CHECKPOINT.is_file():
    checkpoint = torch.load(SOURCE_CHECKPOINT, map_location=device, weights_only=True)
    if checkpoint.get('input_features') != all_features:
        raise ValueError('Checkpoint feature order differs from the current feature schema')
    if checkpoint.get('class_names') != CLASS_NAMES:
        raise ValueError('Checkpoint attack categories do not match')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    print('Loaded saved UNSW source encoder:', SOURCE_CHECKPOINT.name)
else:
    model = train_source_encoder(model)

## 4. Build the adapted model inputs

The 32 neural features are scaled using UNSW training data only. We combine them with the 43 flow features to give the target classifier 75 inputs.

The RL policy uses the 43 flow features to choose which CIC records to review. The policy sees an attack label only **after** selecting that record.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, features, batch_size=2048):
    model.eval()
    results = []
    for start in range(0, len(features), batch_size):
        xb = torch.as_tensor(features[start:start + batch_size], dtype=torch.float32, device=device)
        _, embeddings = model(xb)
        results.append(embeddings.cpu().numpy())
    return np.vstack(results)

z_source = extract_embeddings(model, X_train)
z_adapt = extract_embeddings(model, X_adapt)
z_eval = extract_embeddings(model, X_eval)

embedding_scaler = RobustScaler(quantile_range=(10, 90))
embedding_scaler.fit(np.arcsinh(z_source))

def normalize_embeddings(z):
    return np.clip(embedding_scaler.transform(np.arcsinh(z)), -8, 8).astype(np.float32)

X_pool75 = np.column_stack([X_adapt, normalize_embeddings(z_adapt)]).astype(np.float32)
X_dev75 = np.column_stack([X_eval, normalize_embeddings(z_eval)]).astype(np.float32)

assert X_pool75.shape == (len(target_adapt), 75)
assert X_dev75.shape == (len(target_eval_labels), 75)
assert np.isfinite(X_pool75).all() and np.isfinite(X_dev75).all()
joblib.dump(embedding_scaler, RUN_DIR / 'source_embedding_scaler.joblib')
print('Classifier features:', X_pool75.shape[1])

## 5. Compare how target labels are selected

Each method starts with the same 50 examples and can review 1,000 CIC records in total. Standard RL rewards selecting attacks. Diversity-aware RL gives a larger reward for attack types that have been seen less often. Random selection provides a baseline.

We use seeds 51, 73, and 97. The target evaluation labels are not used to train the policy or classifier.

In [ ]:
SEEDS = (51, 73, 97)
LABEL_BUDGET = 1000
EPSILON = 0.20
initial = np.random.default_rng(78643).choice(
    len(target_adapt), size=50, replace=False
)

pool_binary = target_adapt['Label'].to_numpy(dtype=int)
pool_attack_names = target_adapt['Attack'].fillna('Unknown_Attack').astype(str).str.strip().to_numpy()
dev_binary = target_eval_labels['Label'].to_numpy(dtype=int)
dev_attack_names = np.where(
    dev_binary == 0, 'Benign',
    target_eval_labels['Attack'].fillna('Unknown_Attack').astype(str).str.strip().to_numpy()
)
dev_categories = sorted(set(dev_attack_names) - {'Benign'})
policy_features = torch.as_tensor(X_adapt, dtype=torch.float32, device=device)


def create_policy():
    return nn.Sequential(nn.Linear(43, 32), nn.Tanh(), nn.Linear(32, 1)).to(device)


def acquire_labels(seed, starting_weights, diversity):
    torch.manual_seed(seed + 100)
    rng = np.random.default_rng(seed + 100)
    policy = create_policy()
    policy.load_state_dict(starting_weights)
    optimizer = torch.optim.Adam(policy.parameters(), lr=0.0003)
    available = np.ones(len(pool_binary), dtype=bool)
    available[initial] = False
    selected = list(initial)
    category_counts = Counter(
        pool_attack_names[idx] for idx in initial if pool_binary[idx] == 1
    )
    baseline = 0.0

    while len(selected) < LABEL_BUDGET:
        candidates = rng.choice(np.flatnonzero(available), size=256, replace=False)
        logits = policy(policy_features[candidates]).flatten()
        probabilities = (1 - EPSILON) * torch.softmax(logits, dim=0) + EPSILON / len(candidates)
        distribution = Categorical(probs=probabilities)
        action = distribution.sample()
        index = int(candidates[action.item()])

        # The selected label becomes available only after this choice.
        if pool_binary[index] == 1:
            category = pool_attack_names[index]
            previous = category_counts[category]
            reward = 2 + 6 / np.sqrt(1 + previous) if diversity else 2.0
            category_counts[category] += 1
        else:
            reward = -1.0

        loss = -(reward - baseline) * distribution.log_prob(action) - 0.001 * distribution.entropy()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 2.0)
        optimizer.step()
        baseline = 0.95 * baseline + 0.05 * reward
        selected.append(index)
        available[index] = False

    return np.asarray(selected, dtype=np.int64)


def select_random(seed):
    rng = np.random.default_rng(seed + 100)
    remaining = np.setdiff1d(np.arange(len(pool_binary)), initial)
    return np.concatenate([
        initial, rng.choice(remaining, size=LABEL_BUDGET - len(initial), replace=False)
    ])

print('Label budget:', LABEL_BUDGET, '| Shared starting records:', len(initial))

In [ ]:
def fit_adapted_classifier(indices):
    indices = np.asarray(indices, dtype=np.int64)
    if len(indices) != LABEL_BUDGET or len(np.unique(indices)) != LABEL_BUDGET:
        raise ValueError('Selection does not match the fixed label budget')
    binary = pool_binary[indices]
    names = np.where(binary == 0, 'Benign', pool_attack_names[indices])
    attack_counts = pd.Series(names[binary == 1]).value_counts()
    supported = set(attack_counts[attack_counts >= 3].index)
    train_names = np.array([
        name if (name == 'Benign' or name in supported) else 'Other_Attack'
        for name in names
    ])
    counts = pd.Series(train_names).value_counts()
    weight_map = {
        name: min(5.0, (counts.max() / count) ** 0.25)
        for name, count in counts.items()
    }
    weights = np.array([weight_map[name] for name in train_names])
    classifier = LogisticRegression(solver='lbfgs', max_iter=1200, random_state=13)
    classifier.fit(X_pool75[indices], train_names, sample_weight=weights)
    return classifier, supported


def evaluate_adaptation(classifier, supported, indices):
    probabilities = classifier.predict_proba(X_dev75)
    predicted = classifier.classes_[probabilities.argmax(axis=1)]
    benign_index = list(classifier.classes_).index('Benign')
    attack_scores = 1.0 - probabilities[:, benign_index]
    attacks = dev_binary == 1
    return {
        'Macro_F1': f1_score(
            dev_attack_names, predicted, labels=dev_categories,
            average='macro', zero_division=0
        ),
        'Binary_PR_AUC': average_precision_score(dev_binary, attack_scores),
        'Exact_Type_Accuracy': float(np.mean(predicted[attacks] == dev_attack_names[attacks])),
        'Reviewed_Attacks': int(pool_binary[indices].sum()),
        'Supported_Types': len(supported),
    }


rows = []
saved_selections = {}
for seed in SEEDS:
    torch.manual_seed(seed)
    initial_weights = copy.deepcopy(create_policy().state_dict())
    methods = {
        'Random': select_random(seed),
        'Standard_RL': acquire_labels(seed, initial_weights, diversity=False),
        'Diversity_Aware_RL': acquire_labels(seed, initial_weights, diversity=True),
    }
    for method, chosen in methods.items():
        classifier, supported = fit_adapted_classifier(chosen)
        metrics = evaluate_adaptation(classifier, supported, chosen)
        rows.append({'Seed': seed, 'Method': method, **metrics})
        saved_selections[(seed, method)] = chosen
        print(f'{seed} | {method}: macro-F1={metrics["Macro_F1"]:.4f}, PR-AUC={metrics["Binary_PR_AUC"]:.4f}')

result_df = pd.DataFrame(rows)
display(result_df.round(4))
print('\nMean results:')
display(result_df.groupby('Method')[['Macro_F1', 'Binary_PR_AUC', 'Exact_Type_Accuracy']].mean().round(4))
result_df.to_csv(RUN_DIR / 'adaptation_three_seed_results.csv', index=False)

## 6. Save the adapted classifier

We save the diversity-aware classifier from seed 51 and the exact indices of its 1,000 reviewed CIC records. This creates a fixed model for later evaluation. **Do not use holdout labels to change its settings.**

In [ ]:
chosen = saved_selections[(51, 'Diversity_Aware_RL')]
frozen_classifier, supported = fit_adapted_classifier(chosen)
joblib.dump(frozen_classifier, RUN_DIR / 'adapted_classifier_seed51.joblib')
np.save(RUN_DIR / 'selected_indices_seed51.npy', chosen)

run_manifest = {
    'source_dataset': 'NF-UNSW-NB15-v2',
    'target_dataset': 'NF-CSE-CIC-IDS2018-v2',
    'feature_schema': SCHEMA_FILE.name,
    'source_checkpoint': SOURCE_CHECKPOINT.name if (LOAD_EXISTING_TEACHER and SOURCE_CHECKPOINT.is_file()) else 'source_encoder_checkpoint.pt',
    'source_split_seed': 13,
    'target_split_seed': 786,
    'initial_label_seed': 78643,
    'selection_seeds': list(SEEDS),
    'selected_seed': 51,
    'label_budget': LABEL_BUDGET,
    'initial_labels': len(initial),
    'min_samples_per_attack_type': 3,
    'classifier_features': 75,
    'evaluation_partition': 'CIC development; duplicate and pool-overlap filtered',
    'independent_holdout_used': False,
    'note': 'These are reproduction-run outputs; published experiment artifacts remain unchanged.'
}
with (RUN_DIR / 'run_config.json').open('w', encoding='utf-8') as f:
    json.dump(run_manifest, f, indent=2)

print('Saved reproduction-run results:', RUN_DIR)
print('Source encoder:', run_manifest['source_checkpoint'])
print('Selected CIC records:', len(chosen))
print('Supported attack categories:', len(supported))
print('Separate CIC holdout accessed: NO')